# SuperGemma4 26B MLX on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kingwabg/supergemma4-colab-runner/blob/main/notebooks/SuperGemma4_Colab_MLX_CUDA.ipynb)

이 노트북은 Mac mini 로컬 메모리 한계를 피하기 위해 Colab GPU 런타임에서 `Jiunsong/supergemma4-26b-uncensored-mlx-4bit-v2`를 직접 실행해보는 실험 경로입니다.

실행 전 Colab 메뉴에서 `Runtime > Change runtime type > Hardware accelerator > GPU`를 선택하세요.

주의할 점:

- 무료 Colab GPU 종류와 사용 시간은 보장되지 않습니다.
- 26B 4-bit 모델은 GPU 메모리가 작으면 OOM이 날 수 있습니다.
- MLX CUDA 경로를 사용합니다. Colab 런타임/드라이버와 맞지 않으면 유료 GPU나 다른 변환 모델이 필요할 수 있습니다.
- 비밀키, 개인정보, 민감한 업무 데이터를 프롬프트에 넣지 마세요.


In [ ]:
# 1. GPU 런타임 확인
import platform
import subprocess
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)


## MLX CUDA 설치

MLX는 Linux CUDA 백엔드를 지원합니다. Colab GPU가 NVIDIA/CUDA 조건을 만족하면 아래 설치로 바로 시도합니다.


In [ ]:
# 2. 런타임 패키지 설치
%pip install -q -U "mlx[cuda12]" mlx-lm huggingface_hub


In [ ]:
# 3. MLX CUDA 사용 가능 여부 확인
import mlx
import mlx.core as mx

print('MLX:', getattr(mlx, '__version__', 'unknown'))
print('CUDA available:', mx.cuda.is_available())

if not mx.cuda.is_available():
    raise RuntimeError('MLX CUDA를 사용할 수 없습니다. Colab GPU/드라이버/런타임 조건을 확인하세요.')


## 모델 설정

처음 실행할 때 Hugging Face에서 모델을 내려받습니다. Colab 런타임이 재시작되면 다시 받을 수 있습니다.

만약 Hugging Face 권한 오류가 나면, 원본 모델 페이지에서 라이선스/접근 조건을 먼저 확인한 뒤 `huggingface_hub.login()`으로 토큰을 입력하세요.


In [ ]:
# 선택: Hugging Face 로그인이 필요할 때만 주석을 해제하세요.
# from huggingface_hub import login
# login()

MODEL_ID = 'Jiunsong/supergemma4-26b-uncensored-mlx-4bit-v2'

# GPU 메모리가 작을수록 낮게 잡습니다. 512는 품질보다 생존성을 우선한 설정입니다.
MAX_KV_SIZE = 512
MAX_TOKENS = 96
TEMPERATURE = 0.6
TOP_P = 0.9

PROMPT = '''당신은 한국어로 짧고 정확하게 답하는 로컬 LLM입니다.
질문: 자기소개를 한 문단으로 해줘.
답변:'''


## 단발 생성 테스트

이 셀은 `mlx_lm.generate` CLI를 사용합니다. 실패하면 아래 OOM 대응 셀의 값을 먼저 낮추세요.


In [ ]:
# 4. 모델 실행
import shutil
import subprocess

binary = shutil.which('mlx_lm.generate')
if binary is None:
    raise RuntimeError('mlx_lm.generate 명령을 찾지 못했습니다. 설치 셀을 다시 실행하세요.')

cmd = [
    binary,
    '--model', MODEL_ID,
    '--prompt', PROMPT,
    '--max-tokens', str(MAX_TOKENS),
    '--temp', str(TEMPERATURE),
    '--top-p', str(TOP_P),
    '--max-kv-size', str(MAX_KV_SIZE),
]

print('Running:', ' '.join(cmd[:4]), '...')
result = subprocess.run(cmd, check=False)
print('exit code:', result.returncode)

if result.returncode != 0:
    print('\n실패했습니다. GPU 메모리 부족이면 MAX_KV_SIZE=256, MAX_TOKENS=48로 낮춰 다시 실행하세요.')


## 사용법

여기 Codex 채팅창에 쓰는 것이 아니라, Colab 노트북 안에서 셀을 실행한 뒤 입력창에 질문을 넣습니다.

1. 위 메뉴에서 `Runtime > Change runtime type > GPU`를 선택합니다.
2. `GPU 런타임 확인`, `MLX CUDA 설치`, `MLX CUDA 사용 가능 여부 확인` 셀을 순서대로 실행합니다.
3. `단발 생성 테스트`가 성공하는지 확인합니다.
4. 아래 `대화형 채팅` 셀을 실행합니다.
5. `나:` 입력창에 질문을 쓰고 Enter를 누릅니다. 종료는 `/exit` 또는 `/quit`입니다.


## OOM 대응

무료 Colab에서 GPU 메모리가 부족하면 아래 값을 낮춘 뒤 `모델 실행` 셀을 다시 실행하세요. 그래도 실패하면 이 모델은 해당 런타임에서 무리일 가능성이 큽니다.


In [ ]:
# 더 보수적인 설정
MAX_KV_SIZE = 256
MAX_TOKENS = 48
print('MAX_KV_SIZE:', MAX_KV_SIZE)
print('MAX_TOKENS:', MAX_TOKENS)


## 대화형 채팅

처음 실행할 때 모델을 한 번 로드합니다. 로드가 끝난 뒤 `나:` 입력창에 질문을 넣으면 됩니다. 무료 Colab에서 OOM이 나면 위의 보수적인 설정 셀을 먼저 실행한 뒤 다시 시도하세요.


In [ ]:
# 5. 대화형 채팅
from mlx_lm import generate, load

SYSTEM_PROMPT = '당신은 한국어로 짧고 정확하게 답하는 AI입니다. 모르면 모른다고 말합니다.'

print('모델을 로드합니다. 처음에는 시간이 걸릴 수 있습니다.')
model, tokenizer = load(MODEL_ID)
print('준비 완료. 종료하려면 /exit 또는 /quit 입력')

history = []

def build_prompt(history, user_text):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages.extend(history[-4:])
    messages.append({'role': 'user', 'content': user_text})
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    lines = [f"시스템: {SYSTEM_PROMPT}"]
    for message in messages[1:]:
        role = '사용자' if message['role'] == 'user' else 'AI'
        lines.append(f"{role}: {message['content']}")
    lines.append('AI:')
    return '\n'.join(lines)

while True:
    user_text = input('\n나: ').strip()
    if user_text.lower() in {'/exit', '/quit'}:
        print('종료합니다.')
        break
    if not user_text:
        continue

    prompt = build_prompt(history, user_text)
    answer = generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=MAX_TOKENS,
        temp=TEMPERATURE,
        top_p=TOP_P,
        max_kv_size=MAX_KV_SIZE,
        verbose=False,
    )
    print('\nAI:', answer.strip())
    history.append({'role': 'user', 'content': user_text})
    history.append({'role': 'assistant', 'content': answer.strip()})


## 판단 기준

- 여기서 성공하면 Mac에는 모델을 다시 설치하지 않고 Colab에서 계속 실험하면 됩니다.
- 여기서 OOM이면 무료 Colab GPU로는 부족한 것입니다. L4/A100급 유료 GPU, 더 작은 2-bit 모델, 또는 GGUF/llama.cpp 계열 변환을 검토하세요.
- 이 노트북은 모델을 GitHub에 올리지 않습니다. 가중치는 Hugging Face에서 런타임마다 받아옵니다.
